# CLPsych teacher labeling (stage 3) on Colab — Gemini API

Runs `generate_aux_labels.py` against the Gemini API for the CLPsych auxiliary
schema. No GPU needed — this just calls the hosted model, so the default CPU
runtime is fine.

### What is being labeled
Each post gets predicted `Switch` (0/S), `Escalation` (0/E), `Well-being` (1-10
or null), `adaptive_presence` (1-5 or null), `maladaptive_presence` (1-5 or null).
See `config/clpsych_schema.md` in the repo for the full rubric.

### Ethics reminder
- Post text is sent to the Gemini API. Confirm your institution's data policy allows this.
- Teacher labels are **noisy training supervision**, never clinical ground truth.
- Never send held-out test timelines through this notebook — upload only the
  test-excluded `train_dev_for_teacher.csv` produced locally (see the README's
  Quick start step 2b), not the full `all_with_splits.csv`.

In [ ]:
!pip install -q google-genai pandas tqdm

## Set your Gemini API key
Preferred: add `GEMINI_API_KEY` to Colab's secret manager (key icon in the left
sidebar), grant this notebook access, then run the cell below. If you don't have a
secret configured, it falls back to a hidden prompt. Get a key from
https://aistudio.google.com/app/apikey if you don't have one.

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:
    import getpass
    os.environ["GEMINI_API_KEY"] = getpass.getpass("GEMINI_API_KEY: ")
print("key set:", bool(os.environ.get("GEMINI_API_KEY")))

## Recreate the repo's relative layout
`generate_aux_labels.py` loads the prompt from `../config/teacher_prompt_v1.txt` relative to its own location, so we mirror `src/` and `config/` here.

In [ ]:
import os
for d in ["/content/clpsych/src", "/content/clpsych/config",
          "/content/clpsych/data/processed/splits", "/content/clpsych/data/synthetic_aux"]:
    os.makedirs(d, exist_ok=True)
print("ready")

## Upload the three files
When prompted, upload (from your local `cssrs_pipeline` checkout, after running
`prepare_clpsych.py` + `create_data_splits.py` + the test-exclusion step locally):
1. `src/generate_aux_labels.py`
2. `config/teacher_prompt_v1.txt`
3. `data/processed/splits/train_dev_for_teacher.csv` (train+dev only — test already excluded)

In [ ]:
from google.colab import files
import shutil

print("Upload generate_aux_labels.py ...")
up = files.upload()
shutil.move(next(iter(up)), "/content/clpsych/src/generate_aux_labels.py")

print("Upload teacher_prompt_v1.txt ...")
up = files.upload()
shutil.move(next(iter(up)), "/content/clpsych/config/teacher_prompt_v1.txt")

print("Upload train_dev_for_teacher.csv ...")
up = files.upload()
shutil.move(next(iter(up)), "/content/clpsych/data/processed/splits/train_dev_for_teacher.csv")

print("done")

## Pilot run — 200 posts x 3 runs
Per the README's pilot gate: measure on 200 first, freeze the prompt, then scale to the full set.

In [ ]:
%cd /content/clpsych
!python src/generate_aux_labels.py \
    --input data/processed/splits/train_dev_for_teacher.csv \
    --output data/synthetic_aux/raw_primary_runs.jsonl \
    --model gemini-2.5-flash \
    --runs 3 --limit 200

## Download the result
Bring `raw_primary_runs.jsonl` back to your machine, then run `quality_gates.py` locally to get `pilot_metrics.json` (valid_run_rate, switch_distribution, rejection_reasons) and decide whether to tighten the prompt before the full run.

In [ ]:
from google.colab import files
files.download("/content/clpsych/data/synthetic_aux/raw_primary_runs.jsonl")

## Full run (after the pilot gate passes)
Re-upload the frozen `teacher_prompt_v2.txt` as `config/teacher_prompt_v1.txt` (or edit `PROMPT_PATH` in the script), drop `--limit 200`, and re-run the cell above against the full `train_dev_for_teacher.csv`.